Imports

In [8]:

import pandas as pd
import numpy as np
from pathlib import Path


Rutas

In [9]:
BASE = Path("../../")

HOTEL_EDA = BASE / "data" / "processed" / "livvo_day_hotel_final.parquet"
TTOO_EDA  = BASE / "data" / "processed" / "livvo_day_ttoo_final.parquet"

DASHBOARD_DATA = BASE / "dashboard" / "data"
DASHBOARD_DATA.mkdir(parents=True, exist_ok=True)

print("Hotel EDA file:", HOTEL_EDA)
print("TTOO EDA file:", TTOO_EDA)
print("Saving to:", DASHBOARD_DATA)


Hotel EDA file: ..\..\data\processed\livvo_day_hotel_final.parquet
TTOO EDA file: ..\..\data\processed\livvo_day_ttoo_final.parquet
Saving to: ..\..\dashboard\data


Cargar datasets originales del EDA

In [10]:
df_hotel = pd.read_parquet(HOTEL_EDA)
df_ttoo  = pd.read_parquet(TTOO_EDA)

df_hotel["date"] = pd.to_datetime(df_hotel["date"])
df_ttoo["date"]  = pd.to_datetime(df_ttoo["date"])

Función para resumir comportamiento HOTEL (del EDA por hotel)

In [11]:
def summarize_hotel(df):

    df = df.sort_values("date")

    ocup_mean = df["ocup_total"].mean()
    ocup_std  = df["ocup_total"].std()
    ocup_min  = df["ocup_total"].min()
    ocup_max  = df["ocup_total"].max()

    rn_mean   = df["roomnights"].mean()
    adr_mean  = df["ADR"].mean()

    q1, q3 = df["ocup_total"].quantile([0.25,0.75])
    iqr = q3 - q1
    low  = q1 - 1.5*iqr
    high = q3 + 1.5*iqr
    outlier_ratio = ((df["ocup_total"] < low) | (df["ocup_total"] > high)).mean()

    t = np.arange(len(df))
    slope = float(np.polyfit(t, df["ocup_total"], 1)[0])

    weekly_std  = df.groupby(df["date"].dt.dayofweek)["ocup_total"].mean().std()
    monthly_std = df.groupby(df["date"].dt.month)["ocup_total"].mean().std()

    season_week_strength  = weekly_std / ocup_std if ocup_std>0 else 0
    season_month_strength = monthly_std / ocup_std if ocup_std>0 else 0

    corr_rn  = df["roomnights"].corr(df["ocup_total"])
    corr_adr = df["ADR"].corr(df["ocup_total"])

    return {
        "ocup_mean": ocup_mean,
        "ocup_std": ocup_std,
        "ocup_min": ocup_min,
        "ocup_max": ocup_max,
        "roomnights_mean": rn_mean,
        "adr_mean": adr_mean,
        "outlier_ratio": outlier_ratio,
        "trend_slope": slope,
        "weekly_seasonality_strength": season_week_strength,
        "monthly_seasonality_strength": season_month_strength,
        "corr_roomnights": corr_rn,
        "corr_adr": corr_adr
    }

Función para resumir TTOO por hotel (del EDA por ttoo)

In [12]:
def summarize_ttoo(df):

    pivot = (
        df.groupby(["codigo_hotel","codigo_ttoo"])["roomnights"]
          .sum()
          .unstack(fill_value=0)
    )

    rows = []

    for hotel in pivot.index:

        row = pivot.loc[hotel]
        total = row.sum()
        share = (row / total).sort_values(ascending=False)

        dominant = share.index[0]
        dom_share = float(share.iloc[0])

        p = share[share>0].values
        entropy = float(-(p*np.log(p)).sum())

        top5 = share.head(5).to_dict()

        rows.append({
            "hotel": hotel,
            "dominant_ttoo": dominant,
            "dominant_share": dom_share,
            "entropy_mix": entropy,
            "top5_ttoo_shares": top5
        })

    return pd.DataFrame(rows)

Generar summary final unificando HOTEL + TTOO

In [13]:
hotels = df_hotel["codigo_hotel"].unique()

summary_rows = []

for hotel in hotels:

    df_h = df_hotel[df_hotel["codigo_hotel"] == hotel]
    df_t = df_ttoo [df_ttoo ["codigo_hotel"] == hotel]

    hotel_stats = summarize_hotel(df_h)
    ttoo_stats  = summarize_ttoo(df_t)[summary_rows.__len__():]

    row = {"hotel": hotel}
    row.update(hotel_stats)

    t_row = summarize_ttoo(df_t).iloc[0].to_dict()
    row.update({
        "dominant_ttoo": t_row["dominant_ttoo"],
        "dominant_share": t_row["dominant_share"],
        "entropy_mix": t_row["entropy_mix"],
        "top5_ttoo_shares": t_row["top5_ttoo_shares"]
    })

    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
df_summary

,hotel,ocup_mean,ocup_std,ocup_min,ocup_max,roomnights_mean,adr_mean,outlier_ratio,trend_slope,weekly_seasonality_strength,monthly_seasonality_strength,corr_roomnights,corr_adr,dominant_ttoo,dominant_share,entropy_mix,top5_ttoo_shares
0,HOTEL_1,0.475326,0.141696,0.031915,0.840426,44.680664,97.451829,0.004883,0.000045,0.208414,0.319370,1.0,0.031018,WEL,0.380915,1.799836,"{'WEL': 0.38091491268332134, 'B': 0.2395471335..."
1,HOTEL_2,0.677181,0.148720,0.210145,0.905797,93.451023,150.331019,0.032293,0.000054,0.076469,0.790040,1.0,0.739864,T,0.372892,1.823154,"{'T': 0.37289209362329523, 'J': 0.181717655731..."
2,HOTEL_3,0.640656,0.093633,0.342629,0.820717,160.804608,173.907532,0.016590,0.000001,0.024920,0.811998,1.0,0.408228,J,0.283889,2.132513,"{'J': 0.2838891977555266, 'T': 0.1459194259283..."


Guardar dataset final

In [14]:
df_summary.to_parquet(DASHBOARD_DATA / "eda_summary.parquet", index=False)
print("✅ EDA summary guardado en:", DASHBOARD_DATA / "eda_summary.parquet")

✅ EDA summary guardado en: ..\..\dashboard\data\eda_summary.parquet
